<a href="https://colab.research.google.com/github/CeZa18/ITAI_1372_Labs/blob/main/ollama_cyber_101_CesarZaldivar_ITAI1372.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ollama is now installed. Ollama is an open-source tool that simplifies running large language models locally.
It provides an easy way to download, run, and manage various LLMs on your machine.

If you encounter any GPU-related issues, try the following troubleshooting steps:
1. Ensure CUDA is properly installed and configured.
2. Check if the correct GPU drivers are installed.
3. Verify that Ollama has access to the GPU resources.

Now, let's start Ollama in a separate thread so we can use it throughout this notebook.

We then start ollama in a separate thread so that we can use it after

Reference: https://stackoverflow.com/a/77828874

In [ ]:
# Install Ollama and System Dependencies

!apt-get update -qq
!apt-get install -y -qq zstd curl

!command -v ollama >/dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh

!command -v ollama
!ollama --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
/usr/local/bin/ollama
ollama version is 0.40.11


In [ ]:
# Install Python Packages
%pip install -q openai pydantic instructor feedparser newspaper3k lxml_html_clean

# Verify Python packages

Add quick verifications

In [ ]:
# Verify Python packages
import requests
import openai
import pydantic
import instructor

print("requests:", requests.__version__)
print("openai:", openai.__version__)
print("pydantic:", pydantic.__version__)
print("instructor imported successfully")

requests: 2.32.4
openai: 2.54.0
pydantic: 2.14.1
instructor imported successfully


## Playing with ollama API

In [ ]:
# Start Ollama

import os
import subprocess
import time
import requests

OLLAMA_URL = "http://127.0.0.1:11434"

def ollama_is_running():
    try:
        response = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=3
        )
        return response.status_code == 200
    except:
        return False

if not ollama_is_running():

    log_file = open("/tmp/ollama.log", "a")

    process = subprocess.Popen(
        ["/usr/local/bin/ollama", "serve"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        env=os.environ.copy()
    )

    print("Starting Ollama with PID:", process.pid)

    for _ in range(15):
        time.sleep(2)

        if ollama_is_running():
            print("Ollama server is ready.")
            break
else:
    print("Ollama already running.")

Ollama already running.


In [ ]:
# We first need to install dependencies
!pip install openai pydantic instructor

For this first example, we will use [gemma2](https://blog.google/technology/developers/google-gemma-2/) from Google in its 9 billion parameters version. Its currently [one of the best small](https://chat.lmsys.org/?leaderboard) (<12B parameters) and opensource model.


In [ ]:
# Check/Pull Gemma2

import subprocess

models = subprocess.run(
    ["/usr/local/bin/ollama", "list"],
    capture_output=True,
    text=True
)

print(models.stdout)

if "gemma2:" not in models.stdout:

    print("Gemma 2 is not present. Downloading it now...")

    subprocess.run(
        ["/usr/local/bin/ollama", "pull", "gemma2"],
        check=True
    )
else:
    print("Gemma2 already installed.")

NAME             ID              SIZE      MODIFIED          
gemma2:latest    ff02c3702f32    5.4 GB    About an hour ago    

Gemma2 already installed.


In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:11434/api/tags",
    timeout=10
)

print("Status:", response.status_code)
print(response.json())

Status: 200
{'models': [{'name': 'gemma2:latest', 'model': 'gemma2:latest', 'modified_at': '2026-10-25T02:31:17.600990142Z', 'size': 5443152417, 'digest': 'ff02c3702f322b9e075e9568332d96c0a7028002f1a5a056e0a6784320a4db0b', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'gemma2', 'families': ['gemma2'], 'parameter_size': '9.2B', 'quantization_level': 'Q4_0', 'context_length': 8192, 'embedding_length': 3584, 'runner': 'ggml'}, 'capabilities': ['completion']}]}


In [ ]:
MODEL = "gemma2"

print("Using model:", MODEL)

Using model: gemma2


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama"
)

In [ ]:
MODEL = "gemma2"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: OpenAI interface working."
        }
    ]
)

print(response.choices[0].message.content)

OpenAI interface working.


## Use Case I: Generating malware information cards

In cybersecurity, having quick access to malware information is crucial for threat analysis and incident response.
We'll use our LLM to generate concise "malware cards" based on malware names. This demonstrates how LLMs can be
used to quickly summarize and present complex security information.

However, it's important to note:
1. This information is generated based on the LLM's training data and may not always be up-to-date or accurate.
2. **Always verify critical information with authoritative sources before making security decisions**.
3. Be cautious about potential hallucinations, especially for lesser-known malware.

In [ ]:
from pydantic import BaseModel
from typing import List
import instructor

In [ ]:
!pip install openai pydantic instructor jsonref

#### Fetch malware data

In [ ]:
# Run one card at a time

from openai import OpenAI
from pydantic import BaseModel, Field
import instructor
import time

MODEL = "gemma2"


class Malware(BaseModel):
    name: str
    first_seen: str = Field(
        description="Approximate first-seen year or date, or Unknown"
    )
    language: str = Field(
        description="Implementation language, or Unknown"
    )
    architecture: str = Field(
        description="Target architecture, or Unknown"
    )
    developer: str = Field(
        description="Developer or threat actor, or Unknown"
    )
    description: str = Field(
        description="Brief defensive description under 80 words"
    )


openai_client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    timeout=180.0
)

client = instructor.from_openai(
    openai_client,
    mode=instructor.Mode.JSON
)


def get_malware_description(malware_name: str) -> Malware:
    return client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Create a concise defensive malware information card. "
                    "Follow the JSON schema exactly. "
                    "Use Unknown for uncertain values. Do not speculate."
                )
            },
            {
                "role": "user",
                "content": f"Create a malware card for {malware_name}."
            }
        ],
        response_model=Malware,
        max_retries=0,
        temperature=0
    )


start = time.time()

print("Processing WannaCry...")

try:
    result = get_malware_description("WannaCry")
    print(result.model_dump_json(indent=2))
except Exception as error:
    print("Request failed:", repr(error))

print(f"Elapsed time: {time.time() - start:.1f} seconds")

Processing WannaCry...
{
  "name": "WannaCry",
  "first_seen": "2017",
  "language": "Unknown",
  "architecture": "x86",
  "developer": "Lazarus Group",
  "description": "WannaCry is a ransomware worm that exploits a vulnerability in Microsoft Windows. It encrypts files on infected systems and demands payment for decryption."
}
Elapsed time: 274.4 seconds


In [ ]:
!ollama pull llama3.2:3b

In [ ]:
!ollama list

NAME             ID              SIZE      MODIFIED       
llama3.2:3b      a80c4f17acd5    2.0 GB    38 seconds ago    
gemma2:latest    ff02c3702f32    5.4 GB    2 hours ago       


In [ ]:
MODEL = "llama3.2:3b"

In [ ]:
start = time.time()

print("Processing WannaCry with", MODEL)

try:
    result = get_malware_description("WannaCry")
    print(result.model_dump_json(indent=2))
except Exception as error:
    print("Request failed:", repr(error))

print(f"Elapsed time: {time.time() - start:.1f} seconds")

Processing WannaCry with llama3.2:3b
{
  "name": "WannaCry",
  "first_seen": "2017",
  "language": "C",
  "architecture": "Windows",
  "developer": "Unknown",
  "description": "Ransomware attack that spread globally in 2017, targeting Windows operating systems, exploiting the EternalBlue vulnerability."
}
Elapsed time: 79.6 seconds


In [ ]:
malware_names = [
    "WannaCry",
    "Stuxnet",
    "DarkComet"
]

for index, malware_name in enumerate(malware_names, start=1):
    print(f"\nProcessing {index}/{len(malware_names)}: {malware_name}")

    start = time.time()

    try:
        result = get_malware_description(malware_name)
        print(result.model_dump_json(indent=2))
    except Exception as error:
        print(f"Failed: {error}")

    print(f"Elapsed time: {time.time() - start:.1f} seconds")


Processing 1/3: WannaCry
{
  "name": "WannaCry",
  "first_seen": "2017",
  "language": "C",
  "architecture": "x86",
  "developer": "Unknown",
  "description": "A ransomware attack that spread globally in 2017, targeting Windows operating systems, and encrypting files to demand payment in Bitcoin."
}
Elapsed time: 31.2 seconds

Processing 2/3: Stuxnet
{
  "name": "Stuxnet",
  "first_seen": "2009",
  "language": "C",
  "architecture": "x86",
  "developer": "Unknown",
  "description": "A highly sophisticated computer worm and Trojan horse that targeted industrial control systems, particularly those used in Iran's nuclear program."
}
Elapsed time: 29.1 seconds

Processing 3/3: DarkComet
{
  "name": "DarkComet",
  "first_seen": "2007",
  "language": "C++",
  "architecture": "x86",
  "developer": "Unknown",
  "description": "A remote access trojan (RAT) that allows attackers to control infected computers, steal data, and spread malware."
}
Elapsed time: 28.8 seconds


## Use Case II: Building a daily cybersecurity news digest

Staying informed about the latest cybersecurity developments is essential but time-consuming. This use case
demonstrates how to leverage LLMs to create a personalized, AI-curated news digest.

Benefits:
1. Time-saving: Quickly get an overview of the most important cybersecurity news.
2. Prioritization: Focus on high-impact stories first.
3. Contextual understanding: Get AI-generated summaries and potential implications of each story.

The process involves:
1. Fetching articles from trusted RSS feeds
2. Using the LLM to rank and summarize the articles
3. Generating a comprehensive digest with actionable insights

Let's walk through each step:

In [ ]:
!pip install feedparser

#### Fetching of RSS entries

In [ ]:
from pydantic import BaseModel, Field
from datetime import datetime
import uuid

class Article(BaseModel):
    id: str = Field(..., description="Unique identifier for the article")
    title: str = Field(..., description="Title of the article")
    link: str = Field(..., description="URL of the article")
    published: datetime = Field(..., description="Date and time the article was published")

import feedparser
import uuid

RSS_FEEDS = [
    "https://feeds.feedburner.com/TheHackersNews",
    "https://www.bleepingcomputer.com/feed/",
]

# fetch last rss entries and return title, url and id
def fetch_rss_entries(url):
    feed = feedparser.parse(url)
    return [Article(id=uuid.uuid4().hex[:8],
                   title=entry.title,
                   link=entry.link,
                   published=datetime(*entry.published_parsed[:6]))
            for entry in feed.entries]

articles = []
for rss_feed in RSS_FEEDS:
  articles.extend(fetch_rss_entries(rss_feed))
articles

[Article(id='bc88d43f', title='Brotli Decoder Flaw Behind Node.js RCE Also Affects Bun, Python and Other Widely Used Software', link='https://thehackernews.com/2026/10/brotli-decoder-flaw-behind-nodejs-rce.html', published=datetime.datetime(2026, 10, 24, 11, 9, 29)),
 Article(id='46bd9118', title='Leaked LiteLLM Master Key Let Attackers Steal GitHub Tokens and Hijack Release Pipeline', link='https://thehackernews.com/2026/10/leaked-litellm-master-key-let-attackers.html', published=datetime.datetime(2026, 10, 24, 11, 3, 33)),
 Article(id='a333e684', title='Google Says Gmail and Photos Data Shared With AI Search Could Be Used for Targeted Ads', link='https://thehackernews.com/2026/10/google-says-gmail-and-photos-data.html', published=datetime.datetime(2026, 10, 24, 10, 36, 33)),
 Article(id='30d30af1', title='Researchers Push Over 2 Million Updates to Remotely Disable Popcorn Time Updater That Could Fuel a Botnet', link='https://thehackernews.com/2026/10/researchers-push-over-2-million-u

#### Ranking using LLM

In [ ]:
print("Total articles:", len(articles))

for article in articles[:5]:
    print(article.id, "-", article.title)

Total articles: 50
bc88d43f - Brotli Decoder Flaw Behind Node.js RCE Also Affects Bun, Python and Other Widely Used Software
46bd9118 - Leaked LiteLLM Master Key Let Attackers Steal GitHub Tokens and Hijack Release Pipeline
a333e684 - Google Says Gmail and Photos Data Shared With AI Search Could Be Used for Targeted Ads
30d30af1 - Researchers Push Over 2 Million Updates to Remotely Disable Popcorn Time Updater That Could Fuel a Botnet
3dabe96d - CISA Flags Two Actively Exploited Critical Flaws in Collabora Online and Arista VeloCloud


In [ ]:
# Reduced number of articles to rank to optimize running the model

from typing import List
import time

from openai import OpenAI
from pydantic import BaseModel, Field
import instructor


MODEL = "llama3.2:3b"
MAX_CANDIDATES = 15     # Adjusting articles to 15 (Run 1)


class ArticleRef(BaseModel):
    id: str
    relevance_score: int = Field(ge=1, le=10)


class RankedCyberNews(BaseModel):
    title: str = Field(max_length=100)
    summary: str = Field(max_length=250)
    impact_score: int = Field(ge=1, le=10)
    related_articles: List[ArticleRef] = Field(min_length=1)


class RankedCyberNewsResponse(BaseModel):
    news: List[RankedCyberNews] = Field(
        min_length=3,
        max_length=3
    )


openai_client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    timeout=300.0
)

ranking_client = instructor.from_openai(
    openai_client,
    mode=instructor.Mode.JSON
)


def prepare_candidates(articles, limit=MAX_CANDIDATES):
    candidates = []
    seen_titles = set()

    for article in articles:
        title = str(article.title).strip()
        normalized_title = title.lower()

        if not title or normalized_title in seen_titles:
            continue

        seen_titles.add(normalized_title)
        candidates.append(article)

        if len(candidates) >= limit:
            break

    return candidates


def rank_cyber_news(articles) -> List[RankedCyberNews]:
    candidates = prepare_candidates(articles)

    print(
        f"Ranking {len(candidates)} candidates "
        f"from {len(articles)} collected articles..."
    )

    article_data = "\n".join(
        f"{article.id} | {str(article.title)[:160]}"
        for article in candidates
    )

    prompt = f"""
Select exactly three significant cybersecurity news items from the
candidate list.

For each item:
1. Write a title under 100 characters.
2. Write a summary under 250 characters.
3. Assign an impact score from 1 to 10.
4. Include at least one supplied article ID.
5. Assign each cited ID a relevance score from 1 to 10.

Prioritize:
- Actively exploited vulnerabilities
- Remote-code-execution vulnerabilities
- Major breaches or supply-chain compromise
- Large-scale malware or botnet activity
- Events affecting widely used technology

Use only the supplied titles and IDs.
Do not invent IDs or events.
Return exactly three ranked news items.

Candidates:
{article_data}
"""

    start_time = time.time()

    response = ranking_client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a defensive cybersecurity analyst. "
                    "Rank supplied news titles by likely operational impact. "
                    "Follow the JSON schema exactly."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        response_model=RankedCyberNewsResponse,
        max_retries=0,
        temperature=0,
        max_tokens=750
    )

    elapsed = time.time() - start_time
    print(f"Ranking completed in {elapsed:.1f} seconds.")

    return response.news


ranked_news = rank_cyber_news(articles)

for index, news_item in enumerate(ranked_news, start=1):
    print(f"\nRank {index}")
    print(news_item.model_dump_json(indent=2))

Ranking 15 candidates from 50 collected articles...
Ranking completed in 456.5 seconds.

Rank 1
{
  "title": "Brotli Decoder Flaw Behind Node.js RCE Also Affects Bun, Python and Other Widely Used Software",
  "summary": "A critical vulnerability in Brotli decoder could be exploited for remote code execution in Node.js, Bun, Python, and other software.",
  "impact_score": 9,
  "related_articles": [
    {
      "id": "bc88d43f",
      "relevance_score": 10
    }
  ]
}

Rank 2
{
  "title": "Critical Flaw in Python’s Popular Pillow Image Library Allows Code Execution via a Crafted Image",
  "summary": "A critical vulnerability in Python's Pillow image library could be exploited for code execution via a crafted image.",
  "impact_score": 8,
  "related_articles": [
    {
      "id": "28227e41",
      "relevance_score": 9
    }
  ]
}

Rank 3
{
  "title": "BlackX UEFI Bootkit Bypasses Secure Boot on Patched Windows Systems Missing Revocations",
  "summary": "A new UEFI bootkit could bypass sec

In [ ]:
# For each subject, we will parse articles using newspaper lib and ask for an actionnable summary
!pip install newspaper4k

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [newspaper4k]


#### Article scraping

In [ ]:
from newspaper import Article as ArticleParser

full_text = ""

for ranked_article in ranked_news:
    full_text += f"# News: {ranked_article.title}\n\n"
    full_text += f"Summary: {ranked_article.summary}\n\n"
    full_text += f"Impact Score: {ranked_article.impact_score}\n\n"

    # Select the article with the highest relevance score
    if ranked_article.related_articles:
        most_relevant_article = max(ranked_article.related_articles, key=lambda x: x.relevance_score)

        # Get article link using id
        article_link = next((a.link for a in articles if a.id == most_relevant_article.id), None)
        if article_link:
            full_text += f"# Most Relevant Article (Relevance: {most_relevant_article.relevance_score}):\n\n{article_link}\n\n"
            try:
                article_obj = ArticleParser(article_link)
                article_obj.download()
                article_obj.parse()
                full_text += f"# Text:\n\n{article_obj.text}\n\n"  # Limit to first 1000 characters
            except Exception as e:
                full_text += f"Error fetching article: {str(e)}\n\n"
    else:
        full_text += "No related articles found for this news item.\n\n"

print(f"Processed {len(ranked_news)} news items.")

Processed 3 news items.


#### Digest building

In [ ]:
from openai import OpenAI

def summarize_articles(full_text: str) -> str:
    client = OpenAI(
        base_url="http://127.0.0.1:11434/v1",
        api_key="ollama",  # required, but unused
    )
    prompt = f"""
    As an expert cybersecurity analyst, create a concise yet comprehensive daily news digest in markdown format. Your task:

    1. Analyze the provided cybersecurity articles.
    2. Summarize the given stories, which are already ranked by importance.
    3. For each story, include:
       - The provided headline
       - A brief summary (2-3 sentences) highlighting key points
       - The potential impact on individuals or organizations (consider the given impact score)
       - Key details from the most relevant related article
       - Actionable advice or takeaways for readers

    4. Use bullet points for clarity and readability.
    5. Conclude with a "Bottom Line" that provides an overall assessment of the day's cybersecurity landscape.

    Guidelines:
    - Use clear, professional language avoiding unnecessary jargon.
    - Prioritize stories based on their impact scores.
    - Ensure the total digest length is between 500-750 words.
    - Use markdown formatting to enhance readability (e.g., headers, bold for emphasis).

    Articles to summarize:

    {full_text}

    Produce only the markdown-formatted digest without any additional commentary or metadata.
    """
    return client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    ).choices[0].message.content

digest = summarize_articles(full_text)
print(digest)

# CVE-2023-12977: Multiple Vulnerabilities in Pillow Library

## Summary

A series of vulnerabilities were discovered in the Pillow Python library, allowing attackers to run arbitrary code.

## Impacted Versions

The vulnerabilities affect Pillow versions 2.6.0 to 12.3.0.

## Fixed Versions

Pillow 13.0.0 fixes all the vulnerabilities, and users are advised to upgrade to the latest version.

## Fixes

Several fixes have been implemented:

*   # CVE-2026-112403: A bug where a memory buffer is overflowed when creating thumbnails of crafted lossless JPEG images.
*   # CVE-2026-112393: A vulnerability that overflows a memory buffer when saving compressed TIFF images to memory.
*   # CVE-2026-112401: A memory bug where stale image data is reused after a mode change, exposing it to arbitrary code execution.
*   # CVE-2026-112402: Two overflow flaws in JPEG decoding when dealing with FlashPix (FPX) format files.

## Impact

Attacks require no user interaction and exploit the library to mainta

#### Bonus: build your daily podcast

In this part, we will use another prompt to generate the script for a podcast about the last cybersecurity news. We will then use

In [ ]:
from openai import OpenAI

podcast_client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    timeout=180.0
)

def build_podcast(ranked_news):

    content = ""

    for item in ranked_news:
        content += (
            f"Title: {item.title}\n"
            f"Summary: {item.summary}\n"
            f"Impact Score: {item.impact_score}\n\n"
        )

    prompt = f"""
Create a cybersecurity podcast script.

Requirements:
- Use the three stories provided.
- 250-350 words.
- Friendly conversational tone.
- Explain why each story matters.
- Include simple defensive advice.
- End with a brief sign-off.

Stories:

{content}

Return only the script.
"""

    response = podcast_client.chat.completions.create(
        model=MODEL,
        temperature=0,
        max_tokens=300,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

podcast = build_podcast(ranked_news)

print(podcast)

Here is the script for the cybersecurity podcast:

[Intro music plays]

Host: Welcome to "Cybersecurity Today", the podcast where we dive into the latest threats and vulnerabilities. I'm your host, [Name], and today we're talking about three critical security stories that you need to know about.

First up, we have a vulnerability in the Brotli decoder that affects not just Node.js, but also Bun, Python, and other widely used software. This flaw could be exploited for remote code execution, which means an attacker could take control of your system. Why does this matter? Because it's a critical vulnerability that could be exploited by anyone with access to your system. So, what can you do to protect yourself? Make sure you're running the latest updates and patches for your software, and be cautious when opening attachments or clicking on links from unknown sources.

Next, we have a critical flaw in Python's popular Pillow image library. This vulnerability could be exploited for code exec

In [ ]:
import sys

print("Python:", sys.version)

try:
    from melo.api import TTS
    print("MeloTTS import succeeded.")
except Exception as error:
    print("MeloTTS import failed:", repr(error))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
MeloTTS import failed: ModuleNotFoundError("No module named 'cn2an'")


In [ ]:
with open("/content/podcast_script.txt", "w", encoding="utf-8") as file:
    file.write(podcast)

print("Saved podcast_script.txt")

Saved podcast_script.txt


In [2]:
from google.colab import files
files.download("/content/podcast_script_final_version.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# Load the podcast_script.txt back into memory
with open("/content/podcast_script_final_version.txt", "r", encoding="utf-8") as file:
    podcast = file.read()

print("Podcast script loaded succesfully.")
print("Characters: ", len(podcast))
print()
print(podcast[:500])

Podcast script loaded succesfully.
Characters:  2447

Welcome to "Cybersecurity Today," the podcast where we dive into the latest threats and vulnerabilities. I'm your host, and today we're talking about three critical security stories that you need to know about.
First up, we have a vulnerability in the Brotli decoder that affects not just Node.js, but also Bun, Python, and other widely used software. This flaw could potentially lead to remote code execution, which makes it important for organizations that depend on affected software. What can you


In [9]:
# Verify everything is set up before re-running the notebook
import os

print("File exists:", os.path.exists("/content/podcast_script_final_version.txt"))
print("Podcast variable exists:", "podcast" in globals())
print("Podcast variable type:", type(podcast))
print("Podcast variable length:", len(podcast))

File exists: True
Podcast variable exists: True
Podcast variable type: <class 'str'>
Podcast variable length: 2447


In [ ]:
'''
Freeze this cell before restarting re-installation of MeloTTS.

%cd /content

!git clone https://github.com/myshell-ai/MeloTTS.git
%cd MeloTTS
!pip install -e .
!python -m unidic download
'''


Cloning into 'MeloTTS'...
remote: Enumerating objects: 432, done.
remote: Total 432 (delta 0), reused 0 (delta 0), pack-reused 432 (from 1)
Receiving objects: 100% (432/432), 6.07 MiB | 15.31 MiB/s, done.
Resolving deltas: 100% (204/204), done.
/content/MeloTTS
Obtaining file:///content/MeloTTS
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 29.0 MB/s  0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing 

In [1]:
import numpy
import torch

print("NumPy:", numpy.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

NumPy: 1.26.4
CUDA available: True
GPU: Tesla T4


In [7]:
# Check current Python version
import sys
print(sys.version)

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [8]:
# Removing incomplete clone and clone a clean copy
%cd /content

import os
import shutil

repo_path = '/content/MeloTTS'

if os.path.exists(repo_path):
    shutil.rmtree(repo_path)
    print(f"Repository '{repo_path}' removed successfully.")
else:
    print(f"Repository '{repo_path}' does not exist.")

!git clone https://github.com/myshell-ai/MeloTTS.git /content/MeloTTS

/content
Repository '/content/MeloTTS' does not exist.
Cloning into '/content/MeloTTS'...
remote: Enumerating objects: 432, done.
remote: Total 432 (delta 0), reused 0 (delta 0), pack-reused 432 (from 1)
Receiving objects: 100% (432/432), 6.07 MiB | 9.90 MiB/s, done.
Resolving deltas: 100% (204/204), done.


In [9]:
# Install build tools and compatible dependencies
!apt-get update -qq
!apt-get install -y -qq build-essential python3-dev mecab libmecab-dev mecab-ipdadic-utf8

%pip install -q "setuptools<81" wheel jedi

%pip install -q \
      cn2an \
      jieba \
      pypinyin \
      eng-to-ipa \
      inflect \
      unidecode \
      anyascii \
      num2words \
      g2p_en \
      cached_path \
      txtsplit \
      unidic \
      unidic-lite \
      "fugashi>=1.4.0" \
      "mecab-python3>=1.0.10"


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
E: Unable to locate package mecab-ipdadic-utf8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
 

In [10]:
# Install MeloTTS without incopatible pinned dependencies
%cd /content/MeloTTS
%pip install -q -e . --no-deps # The --no-deps installs the melo package without replacing the newer fugashi and mecab-python3 versions

/content/MeloTTS
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for melotts (pyproject.toml) ... done


In [11]:
# Now download missing UniDic data
!python -m unidic download

download url: https://cotonoha-dic.s3-ap-northeast-1.amazonaws.com/unidic-3.1.0.zip
Dictionary version: 3.1.0+2021-08-31
unidic-3.1.0.zip: 100% 526M/526M [00:13<00:00, 39.5MB/s]
Finished download.
Downloaded UniDic v3.1.0+2021-08-31 to /usr/local/lib/python3.13/dist-packages/unidic/dicdir


In [12]:
# Verify dictionary
import os
import unidic

mecabrc_path = os.path.join(unidic.DICDIR, "mecabrc")

print("UniDic directory:", unidic.DICDIR)
print("mecabrc path:", mecabrc_path)
print("mecabrc exists:", os.path.exists(mecabrc_path))

UniDic directory: /usr/local/lib/python3.13/dist-packages/unidic/dicdir
mecabrc path: /usr/local/lib/python3.13/dist-packages/unidic/dicdir/mecabrc
mecabrc exists: True


In [13]:
# Clean typo error ipadic
!apt-get update -qq
!apt-get install -y -qq build-essential python3-dev mecab libmecab-dev mecab-ipadic-utf8

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package javascript-common.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../00-javascript-common_11+nmu1_all.deb ...
Unpacking javascript-common (11+nmu1) ...
Selecting previously unselected package libjs-underscore.
Preparing to unpack .../01-libjs-underscore_1.13.4~dfsg+~1.11.4-3_all.deb ...
Unpacking libjs-underscore (1.13.4~dfsg+~1.11.4-3) ...
Selecting previously unselected package libjs-sphinxdoc.
Preparing to unpack .../02-libjs-sphinxdoc_7.2.6-6_all.deb ...
Unpacking libjs-sphinxdoc (7.2.6-6) ...
Selecting previously unselected package libmecab2:amd64.
Preparing to unpack .../03-libmecab2_0.996-14ubuntu4_amd64.deb ...
Unpacking libmecab2:amd64 (0.996-14ubuntu4) ...
Selecting previously unselected package libmecab-

In [14]:
# Install missing pykakasi
%pip install -q pykakasi

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
melotts 0.1.2 requires g2pkk>=0.1.1, which is not installed.
melotts 0.1.2 requires gruut[de,es,fr]==2.2.3, which is not installed.
melotts 0.1.2 requires jamo==0.4.1, which is not installed.
melotts 0.1.2 requires langid==1.1.6, which is not installed.
melotts 0.1.2 requires loguru==0.7.2, which is not installed.
melotts 0.1.2 requires anyascii==0.3.2, but you have anyascii 0.3.3 which is incompatible.
melotts 0.1.2 requires cn2an==0.5.22, but you have cn2an 0.5.24 which is incompatible.
melotts 0.1.2 requires fugashi==1.3.0, but you have fugashi 1.5.2 which is incompatible.
melotts 0.1.2 requires inflect==7.0.0, but you have inflect 7.6.0 which is incompatible.
melotts 0.1.2 requires librosa==0.9.1, but you have librosa 0.11.0 which is incompatible.
melotts 0.1.2 requires mecab-python3==1.0.9, but you have mecab

In [15]:
# Confirm pykakasi installed correctly
import pykakasi

print("pykakasi imported succesfully")

pykakasi imported succesfully


In [21]:
# Install missing packages
# Upgrade NumPy to resolve binary compatibility mismatch first
#%pip install --upgrade -q numpy

%pip install -q \
    "g2pkk>=0.1.1" \
    "gruut[de,es,fr]==2.2.3" \
    "jamo==0.4.1" \
    "langid==1.1.6" \
    "loguru==0.7.2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.11.1 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
contourpy 1.4.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.14.0.94 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
spopt 0.7.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.11.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
xarray-einstats 0.11.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
nx-cugraph-cu13 26.6.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incomp

In [22]:
import importlib.util
import numpy
import networkx

print("NumPy:", numpy.__version__)
print("NetworkX:", networkx.__version__)

packages = ["g2pkk", "gruut", "jamo", "langid", "loguru"]

for package in packages:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package}: {'installed' if installed else 'missing'}")

NumPy: 2.4.6
NetworkX: 2.8.8
g2pkk: installed
gruut: installed
jamo: installed
langid: installed
loguru: installed


In [2]:
# Test MeloTTS import
try:
    from melo.api import TTS
    print("MeloTTS import succeeded.")
except Exception as error:
    print("MeloTTS import failed:")
    print(type(error).__name__ + ":", error)

/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/lib/python3.13/dist-packages/jieba/posseg/__init__.py:16: SyntaxWarning: invalid escape sequence '\.'
  re_skip_detail = re.compile("([\.0-9]+|[a-zA-Z0-9]+)"

config.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/231k [00:00<?, ?B/s]

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

... (16 outputs left out)

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/486k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

MeloTTS import succeeded.


In [23]:
# Verify PyTorch sees the GPU
import torch

print("CUDA available:", torch.cuda.is_available())
print("Podcast loaded:", "podcast" in globals())
print("Model loaded:", "model" in globals())
print("Speaker IDs loaded:", "speaker_ids" in globals())

if "podcast" in globals():
    print("Podcast length:", len(podcast), "characters")

if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("CUDA device name:", torch.cuda.get_device_name(0))

CUDA available: True
Podcast loaded: True
Model loaded: False
Speaker IDs loaded: False
Podcast length: 2447 characters
CUDA device count: 1
CUDA device name: Tesla T4


In [10]:
# TTS test
from melo.api import TTS
from IPython.display import Audio, display

speed = 1.1
device = "cuda:0"

print("Loading MeloTTS on:", device)

model = TTS(
    language="EN",
    device=device
)

speaker_ids = model.hps.data.spk2id
print("Available speakers:", speaker_ids)

test_output = "/content/melo_test.wav"

model.tts_to_file(
    "Welcome to Cybersecurity Today. This is a test of the podcast audio system.",
    speaker_ids["EN-Default"],
    test_output,
    speed=speed
)

print("Test audio created.")
display(Audio(test_output))

Loading MeloTTS on: cuda:0


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Available speakers: {'EN-US': 0, 'EN-BR': 1, 'EN_INDIA': 2, 'EN-AU': 3, 'EN-Default': 4}
 > Text split to sentences.
Welcome to Cybersecurity Today. This is a test of the podcast audio system.
 > ===========================


100%|██████████| 1/1 [00:00<00:00,  2.76it/s]

Test audio created.


In [4]:
# Fix missing nltk package
import nltk

nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [5]:
# Short audio generation test
test_output = "/content/melo_test.wav"

model.tts_to_file(
    "Welcome to Cybersecurity Today. This is a test of the podcast audio system.",
    speaker_ids["EN-Default"],
    test_output,
    speed=1.1
)

print("Test audio created:", test_output)

 > Text split to sentences.
Welcome to Cybersecurity Today. This is a test of the podcast audio system.
 > ===========================


  0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 1/1 [00:19<00:00, 19.55s/it]

Test audio created: /content/melo_test.wav


In [6]:
# Listening test
from IPython.display import Audio, display

display(Audio("/content/melo_test.wav"))

In [ ]:
from melo.api import TTS
# Speed is adjustable
speed = 1.1

# CPU is sufficient for real-time inference.
# You can set it manually to 'cpu' or 'cuda' or 'cuda:0' or 'mps'
# device = 'auto' # Will automatically use GPU if available
device = "cuda:0"

# English
model = TTS(language='EN', device=device)
speaker_ids = model.hps.data.spk2id

# Default accent
output_path = 'en-default.wav'
model.tts_to_file(podcast, speaker_ids['EN-Default'], output_path, speed=speed)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/231k [00:00<?, ?B/s]

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

... (20 outputs left out)

 > Text split to sentences.
Welcome back to edList. io's cyber security news roundup, where we break down the latest threats and how to stay safe. Today's top story is about a sophisticated threat group known as Void Banshee exploiting a newly discovered Microsoft flaw. Cybersecurity researchers at Trend Micro have uncovered an attack campaign where Void Banshee uses a vulnerability in the Microsoft MHTML browser engine to deliver Atlantida,
a powerful information stealer. This vulnerability, tracked as CVE-2024-38112, was actively used by Void Banshee starting in mid-May 2024. What makes this particularly concerning is that Void Banshee exploited the vulnerability through disabled services like Internet Explorer,
highlighting how even seemingly outdated software can pose a risk. The attack chain involves spear phishing emails with links to specially crafted URL files. These files, when opened, exploit the Microsoft MHTML flaw and redirect victims to malicious websites hosting an infec

  0%|          | 0/6 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/conv.py:797: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv_transpose1d(
/usr/local/lib/python3.10/dist-pa

In [ ]:
from IPython.display import Audio #Import Audio method from IPython's Display Class

Audio(output_path, autoplay=False)

In [11]:
# Generate the full podcast
from IPython.display import Audio, display

output_path = "/content/cybersecurity_podcast.wav"

print("Generating full cybersecurity podcast...")

model.tts_to_file(
    podcast,
    speaker_ids["EN-Default"],
    output_path,
    speed=1.1
)

print("Podcast created:", output_path)

display(Audio(output_path))

Generating full cybersecurity podcast...
 > Text split to sentences.
Welcome to Cybersecurity Today, the podcast where we dive into the latest threats and vulnerabilities. I'm your host, and today we're talking about three critical security stories that you need to know about. First up, we have a vulnerability in the Brotli decoder that affects not just Node.
js, but also Bun, Python, and other widely used software. This flaw could potentially lead to remote code execution, which makes it important for organizations that depend on affected software. What can you do to protect yourself? Keep affected software and dependencies updated,
follow vendor security advisories, and apply available security patches as soon as practical. Next, we have a critical flaw affecting Python's popular Pillow image library. The vulnerability involves processing specially crafted images and could potentially result in code execution on affected systems.
This matters because image-processing libraries such a

100%|██████████| 9/9 [00:05<00:00,  1.61it/s]


Podcast created: /content/cybersecurity_podcast.wav


In [12]:
from google.colab import files

files.download("/content/cybersecurity_podcast.wav")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

In this notebook, we've explored how to leverage Large Language Models (LLMs) for cybersecurity applications using Ollama. We've demonstrated two practical use cases:

1. Generating malware information cards for quick reference and analysis.
2. Creating an AI-curated cybersecurity news digest to stay informed about the latest threats and developments.

These examples showcase the potential of LLMs to enhance cybersecurity operations by:
- Quickly synthesizing and presenting complex information
- Automating time-consuming tasks like news curation and summarization
- Providing insights that can aid in threat analysis and decision-making

However, it's crucial to remember that while LLMs are powerful tools, they should be used in conjunction with human expertise and established cybersecurity practices. **Always verify AI-generated information, especially when making critical security decisions**.

Stay curious, stay secure, and keep pushing the boundaries of what's possible in this new field :)